<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Weather Service extraction class - Dev notebook

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

In [ ]:
print(manager.sfd_list.columns)

In [ ]:
#print(manager.sfd_list.columns)
unique_values = manager.sfd_list['field.farm.grower.firstname'].unique()
print(unique_values)


### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from weather_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.extractors.weather_functions import WeatherExtractor
extractor = WeatherExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id", "start_date": "sowingDate"}

extractor.setup_weather_parameters(
    weather_type="HISTORICAL_DAILY",
    weather_parameters="Temperature.standardmax",
    partial_frequency=50,
    exclude_columns=[],
    kpi_filter=None,
    column_mapping=column_mapping
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop.id": "SOYBEANS",
    "start_date":"2025-06-01",
    "end_date":"2025-10-01"
}


#### Test get_weather_api

In [ ]:
print("\n--- Test Case 1: Valid dates ---")
try:
    result = extractor.get_weather_data(
        entity_data=seasonfield_data
    )
    
    print("✅ Weather data retrieved successfully!")
    if isinstance(result, dict) and 'value' in result:
        records = result['value']
        print(f"Number of records: {len(records)}")
        if records:
            print(f"\n📊 First record:")
            for key, value in list(records[0].items())[:5]:
                print(f"  {key}: {value}")
    else:
        print(f"Response: {result}")
    
except Exception as e:
    print(f"❌ API call failed: {e}")
    import traceback
    traceback.print_exc()

#### Test get_weather_data_safe

In [ ]:
print("\n--- Test: get_weather_data_safe ---")
safe_result = extractor.get_weather_data_safe(
        entity_data=seasonfield_data
)
print(safe_result)

#### Test format_weather_json

In [ ]:
print("\n--- Test: format_weather_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_weather_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_weather_json: No valid data from API.")

### 🗺️ process_single_entity

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop":"SOYBEANS",
    "start_date":"2025-06-01",
    "end_date":"2025-10-01",
    'years': [2024, 2023, 2022]
})

result = extractor.process_single_entity_weather(row)

print(result)

### Test historical years validation (list, string, column_mapping)

Validates that `years` works as a native list, comma-separated string (pipeline flattening),
and via `column_mapping` remapping from `historical_seasons`.

In [ ]:
from earthdaily.agriculture.core.api_utils import validate_historical_years

# Test validate_historical_years directly
test_cases = [
    None,
    'ALL',
    [2024, 2023, 2022],
    '2024,2023,2022',
    5,
]

for tc in test_cases:
    result = validate_historical_years(tc)
    print(f'  {str(tc):30s} -> {result} ({type(result).__name__})')


### 🗺️ process_weather_bulk_extraction_parallel

In [ ]:
import pandas as pd
top25 = manager.sfd_list.head(50)

# Convert sowingDate to datetime and compute end_date
top25['sowingDate'] = pd.to_datetime(top25['sowingDate'])
top25['end_date'] = top25['sowingDate'] + pd.Timedelta(days=50)

# Launch extraction with 20 threads 

result = extractor.process_entity_weather_bulk_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="CORN",
    filter_type="exclude" # filter type used to 'include' or 'exclude' row matching column and value filter
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

In [ ]:
print(results.head)


## 🧭 Spatial grouping (geohash dedup) + cache

Weather is queried by field **centroid** (`Location=<centroid>`), so fields whose
centroids fall in the same geohash cell **and** share the same date window return
identical values. `spatial_grouping=True` calls the API **once per
`geohash × start_date × end_date × historical_years` group** and broadcasts to
every member field — big savings at scale (thousands of neighbouring fields →
a few hundred calls), with the same output shape as a per-field run.

- `spatial_precision` — geohash length (default `5` ≈ 4.9 km cells).
- Signature columns are resolved through `column_mapping`, so here `start_date`
  keys on the `sowingDate` column configured above.
- Composes with `use_cache`; result dict gains `representative_calls` / `grouped_from`.

In [ ]:
# Self-contained demo (no platform load needed). Columns follow the configured
# column_mapping: the date window lives in `sowingDate` (mapped from start_date).
# Two co-located fields (same geohash-5 cell) + one ~12 km away (different cell).
import pandas as pd

demo_entities = pd.DataFrame([
    {"id": "grp_a1",
     "geometry": "POLYGON ((-58.9370 -13.7255, -58.9360 -13.7255, -58.9360 -13.7245, -58.9370 -13.7245, -58.9370 -13.7255))",
     "sowingDate": "2025-06-01", "end_date": "2025-10-01"},
    {"id": "grp_a2",  # ~40 m from a1 → same geohash-5 cell
     "geometry": "POLYGON ((-58.9367 -13.7252, -58.9357 -13.7252, -58.9357 -13.7242, -58.9367 -13.7242, -58.9367 -13.7252))",
     "sowingDate": "2025-06-01", "end_date": "2025-10-01"},
    {"id": "grp_b1",  # ~12 km away → different cell
     "geometry": "POLYGON ((-58.8505 -13.6505, -58.8495 -13.6505, -58.8495 -13.6495, -58.8505 -13.6495, -58.8505 -13.6505))",
     "sowingDate": "2025-06-01", "end_date": "2025-10-01"},
])

per_field = extractor.process_entity_weather_bulk_parallel(
    entity_list=demo_entities, skip_export=True, spatial_grouping=False,
)
grouped = extractor.process_entity_weather_bulk_parallel(
    entity_list=demo_entities, skip_export=True,
    spatial_grouping=True, spatial_precision=5,
)

print(f"per-field API calls : {per_field['total_calculations']}")
print(f"grouped API calls   : {grouped['representative_calls']} (from {grouped['grouped_from']} fields)")
print(f"rows: per-field={len(per_field['results_df'])}  grouped={len(grouped['results_df'])}")
assert len(per_field['results_df']) == len(grouped['results_df']), "row-count invariant"
display(grouped['results_df'].head())

In [ ]:
# Cache cold vs warm (grouped + use_cache). Cold run fetches the representatives;
# warm run serves them from the local cache (representative-unit cache stats).
import time

for label in ("cold", "warm"):
    t0 = time.time()
    res = extractor.process_entity_weather_bulk_parallel(
        entity_list=demo_entities, skip_export=True,
        spatial_grouping=True, use_cache=True,
    )
    print(f"{label:>4} run: {time.time() - t0:5.2f}s  "
          f"calls={res['representative_calls']}  "
          f"cache_hit={res.get('cache_hit')}  cache_miss={res.get('cache_miss')}")

### 🗃️ With vs without cache (per-field)

Isolates the cache from spatial grouping so the effect is unambiguous. `use_cache`
resolution is **workflow / per-call `use_cache` > extractor instance default**;
here we pass it per call:

| Run | `use_cache` | Behaviour |
|-----|-------------|-----------|
| no cache | `False` | always calls the API (baseline) |
| cache cold | `True` (empty cache) | misses → fetches → populates cache |
| cache warm | `True` (populated) | full cache hit → **0 API calls**, served from disk |

A fresh temp `cache_dir` is set so the cold run is genuinely cold and the demo is
repeatable. Caching needs a **local** `cache_dir` (remote URIs disable it) and
`cache_key_columns`, which `setup_weather_parameters` already set.

In [ ]:
import time
import tempfile
from pathlib import Path

# Scoped, non-destructive cache dir so 'cold' is genuinely cold and re-runnable.
extractor.cache_dir = Path(tempfile.mkdtemp(prefix="weather_cache_demo_"))
extractor._cache_dir_is_remote = False
print(f"cache dir: {extractor.cache_dir}\n")

runs = [
    ("no cache  ", dict(use_cache=False)),
    ("cache cold", dict(use_cache=True)),
    ("cache warm", dict(use_cache=True)),
]
for label, kw in runs:
    t0 = time.time()
    res = extractor.process_entity_weather_bulk_parallel(
        entity_list=demo_entities, skip_export=True, spatial_grouping=False, **kw,
    )
    print(f"{label}: {time.time() - t0:5.2f}s  "
          f"rows={len(res['results_df'])}  "
          f"cache_hit={res.get('cache_hit', 'n/a')}  cache_miss={res.get('cache_miss', 'n/a')}")

# Expected: cold populates the cache (miss), warm is a full hit (0 API calls) and faster.